# Ablation Study for NativE on Spatial-MMSG
This notebook runs the NativE baseline (Multi-modal Knowledge Graph Completion) on the Spatial-MMSG dataset.
It uses the pre-extracted RGB, Depth, and Text embeddings to prove that Depth improves relation prediction.

In [ ]:
!cp -r /kaggle/input/kaggle-payload/baselines /kaggle/working/
!cp -r /kaggle/input/kaggle-payload/data /kaggle/working/


In [ ]:
%%bash
cd /kaggle/working/baselines/NativE/mmkgc
echo 'Recompiling C++ Data Loader for the current Kaggle CPU architecture...'
sed -i 's/\r$//' make.sh
sed -i 's/mkdir release/mkdir -p release/g' make.sh
bash make.sh


In [ ]:
%%python
import os
# PATCHING THE HARDCODED EMBEDDING SIZES IN NativE'S 3-MODAL ARCHITECTURE
file_path = "/kaggle/working/baselines/NativE/mmkgc/module/model/AdvRelRotatEDB15K.py"
with open(file_path, "r") as f:
    content = f.read()

import re
content = re.sub(
    r"self\.img_dim = 256.*?self\.img_embeddings = nn\.Embedding\.from_pretrained",
    "self.img_dim = img_emb.shape[1]\n        self.text_dim = text_emb.shape[1]\n        self.numeric_dim = numeric_emb.shape[1]\n\n        self.img_embeddings = nn.Embedding.from_pretrained",
    content,
    flags=re.DOTALL
)
# Fix the numeric_proj to use numeric_dim instead of hardcoded text_dim
content = content.replace("self.numeric_proj = nn.Sequential(\n            nn.Linear(self.text_dim, self.dim_e)", 
                          "self.numeric_proj = nn.Sequential(\n            nn.Linear(self.numeric_dim, self.dim_e)")
with open(file_path, "w") as f:
    f.write(content)
print("Patched NativE 3-Modal architecture to support MMSG embedding sizes!")


In [ ]:
!cd /kaggle/working/baselines/NativE && sed -i 's/"link")/"link", type_constrain=False)/g' run_adv_wgan_gp.py
!cd /kaggle/working/baselines/NativE && sed -i 's/"link")/"link", type_constrain=False)/g' run_adv_wgan_gp_3modal.py
!echo 'Running CONTROL NativE (2 Modalities: Visual=RGB, Textual=Text. NO DEPTH)'
!cd /kaggle/working/baselines/NativE && python run_adv_wgan_gp.py -dataset=Spatial-MMSG -batch_size=1024 -margin=12 -epoch=200 -dim=250 -adv_num=1 -save=Spatial-MMSG-Control -neg_num=128 -mu=0.0001 -learning_rate=1e-4 -lrg=1e-4


In [ ]:
!echo 'Running ABLATION 1 (Early Fusion: Visual=Mean(RGB+Depth))'
!cd /kaggle/working/baselines/NativE && cp embeddings/Spatial-MMSG-mean-visual.pth embeddings/Spatial-MMSG-visual.pth
!cd /kaggle/working/baselines/NativE && python run_adv_wgan_gp.py -dataset=Spatial-MMSG -batch_size=1024 -margin=12 -epoch=200 -dim=250 -adv_num=1 -save=Spatial-MMSG-Ablation-Mean -neg_num=128 -mu=0.0001 -learning_rate=1e-4 -lrg=1e-4


In [ ]:
!echo 'Running ABLATION 2 (Late Fusion 3-Modal: Visual=RGB, Numeric=Depth, Textual=Text)'
!cd /kaggle/working/baselines/NativE && cp /kaggle/working/data/Spatial-MMSG/ent_rgb_emb.pt embeddings/Spatial-MMSG-visual.pth
!cd /kaggle/working/baselines/NativE && python run_adv_wgan_gp_3modal.py -dataset=Spatial-MMSG -batch_size=1024 -margin=12 -epoch=200 -dim=250 -adv_num=1 -save=Spatial-MMSG-Ablation-3Modal -neg_num=128 -mu=0.0001 -learning_rate=1e-4 -lrg=1e-4
